In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime
import pytz

# Parámetros del Job
dbutils.widgets.removeAll()
dbutils.widgets.dropdown("env", "dev", ["dev", "prod"], "Ambiente")
dbutils.widgets.text("process_date", "", "Fecha Proceso (DDMMYYYY)")

env = dbutils.widgets.get("env")
process_date = dbutils.widgets.get("process_date")
catalog = f"{env}_fraud"

# Configuración Hora Perú
peru_tz = pytz.timezone('America/Lima')
ts_peru = datetime.now(peru_tz).strftime('%Y-%m-%d %H:%M:%S')

In [0]:
# 3. Lectura y Transformación (LIMPIEZA DE $)
df_bronze = spark.table(f"{catalog}.bronze.clientes_bronze").filter(col("process_date") == process_date)

df_silver = df_bronze.select(
    col("id_cliente").cast("bigint"),
    col("nombre"),
    col("edad_actual").alias("edad_real"),
    when(col("edad_actual") < 18, "Menor").when(col("edad_actual") <= 30, "Joven")
        .when(col("edad_actual") <= 60, "Adulto").otherwise("Senior").alias("rango_edad"),
    col("genero"), col("ciudad"), col("estado"),
    # Limpieza de ingreso y deuda
    regexp_replace(col("ingreso_anual"), r"[\$,]", "").cast("double").alias("ingreso_anual"),
    regexp_replace(col("deuda_total"), r"[\$,]", "").cast("double").alias("deuda_total"),
    col("puntaje_fico")
).select(
    "*",
    when(col("ingreso_anual") < 35000, "Bajo").when(col("ingreso_anual") < 75000, "Medio").otherwise("Alto").alias("rango_ingreso"),
    when(col("puntaje_fico") < 600, "Riesgo").when(col("puntaje_fico") < 750, "Estable").otherwise("Excelente").alias("rango_fico"),
    (col("deuda_total") > (col("ingreso_anual") * 0.5)).alias("flag_alta_deuda"),
    lit(ts_peru).cast("timestamp").alias("ingestion_timestamp"),
    lit(process_date).alias("process_date")
).dropDuplicates(["id_cliente"])

# 4. Upsert (MERGE)
target = f"{catalog}.silver.clientes_silver"
dt = DeltaTable.forName(spark, target)
dt.alias("t").merge(df_silver.alias("s"), "t.id_cliente = s.id_cliente") \
  .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

print(f"✅ Clientes Silver procesados con éxito.")

In [0]:
# 1. Consultamos la tabla física final en el catálogo
df_final_silver = spark.table(target)

# 2. Obtenemos el conteo total acumulado en Silver
total_registros_silver = df_final_silver.count()

# 3. Obtenemos cuántos registros pertenecen al proceso de hoy (opcional)
registros_hoy = df_final_silver.filter(col("process_date") == process_date).count()

print("═" * 70)
print(f" 📊 REPORTE DE TABLA FÍSICA: {target.upper()} ".center(70, "═"))
print("═" * 70)
print(f"{'Catálogo.Esquema.Tabla':<30} : {target}")
print(f"{'Total registros acumulados':<30} : {total_registros_silver:,}")
print(f"{'Registros cargados/actualizados hoy':<30} : {registros_hoy:,}")
print(f"{'Última sincronización (Perú)':<30} : {ts_peru}")
print("═" * 70)

# 4. Lectura de los 3 registros más recientes de la tabla Silver
print(f"\n🔍 VISTA PREVIA DE LA TABLA SILVER (Top 3 recientes):")

# Ordenamos por timestamp para ver lo que acaba de entrar
display(
    df_final_silver
    .orderBy(col("ingestion_timestamp").desc())
    .limit(3)
)